In [2]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 26.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.4 MB/s eta 0:00

In [3]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.6.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [9]:
from huggingface_hub import login

login()

In [12]:
cpt_dataset = load_dataset("KushT/LitCovid_BioCreative", split="train")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-4660a9815f2a5e(…):   0%|          | 0.00/46.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-43d98fd02(…):   0%|          | 0.00/5.01M [00:00<?, ?B/s]

data/test-00000-of-00001-b713fa97ade3c82(…):   0%|          | 0.00/11.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24960 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6239 [00:00<?, ? examples/s]

In [13]:
def format_and_append_eos(example):
    title = str(example.get("title", example.get("Title", "")))
    abst = str(example.get("abstract", example.get("Abstract", example.get("text", ""))))
    
    combined_text = f"Title: {title}\nAbstract: {abst}\n"
    
    return {"text": combined_text + tokenizer.eos_token}

In [14]:
original_columns = cpt_dataset.column_names
cpt_dataset = cpt_dataset.map(format_and_append_eos, remove_columns=original_columns)

Map:   0%|          | 0/24960 [00:00<?, ? examples/s]

In [15]:
cpt_dataset

Dataset({
    features: ['text'],
    num_rows: 24960
})

In [18]:
args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    dataset_text_field = "text",
    max_seq_length = 2048,
    packing = True,
    dataset_num_proc = 3,
)

In [19]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = cpt_dataset,
    args = args,
)

Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/24960 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=3):   0%|          | 0/24960 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [20]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,987 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.925361
2,1.969074
3,1.961897
4,1.691254
5,1.744416
6,1.919612
7,1.807814
8,1.827213
9,1.792478
10,1.956825


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [21]:
adapter_name = "qwen2.5-3b-infectious-disease-lora"
model.save_pretrained(adapter_name)
tokenizer.save_pretrained(adapter_name)

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-infectious-disease-lora/tokenizer_config.json.


('qwen2.5-3b-infectious-disease-lora/tokenizer_config.json',
 'qwen2.5-3b-infectious-disease-lora/tokenizer.json')

In [22]:
!zip -r qwen2.5-3b-infectious-disease-lora.zip /kaggle/working/qwen2.5-3b-infectious-disease-lora

  adding: kaggle/working/qwen2.5-3b-infectious-disease-lora/ (stored 0%)
  adding: kaggle/working/qwen2.5-3b-infectious-disease-lora/tokenizer.json (deflated 81%)
  adding: kaggle/working/qwen2.5-3b-infectious-disease-lora/tokenizer_config.json (deflated 89%)
  adding: kaggle/working/qwen2.5-3b-infectious-disease-lora/README.md (deflated 65%)
  adding: kaggle/working/qwen2.5-3b-infectious-disease-lora/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/qwen2.5-3b-infectious-disease-lora/adapter_config.json (deflated 58%)


In [23]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [24]:
test_prompt = "Title: Efficacy of Antiviral Therapeutics in Severe Acute Respiratory Syndromes\nAbstract: "
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

In [25]:
with model.disable_adapter():
    base_outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
    print(tokenizer.batch_decode(base_outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Title: Efficacy of Antiviral Therapeutics in Severe Acute Respiratory Syndromes
Abstract: 2009 H1N1 influenza pandemic, which was caused by a novel influenza virus, has caused a global pandemic. The pandemic has caused a large number di erent types of respiratory syndromes. The pandemic has caused a large number of deaths, especially in young people. The pandemic has caused a large number of deaths, especially in young people. The pandemic has caused a large number of deaths, especially in young people. The pandemic has caused a large number of deaths, especially in


In [26]:
ft_outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
print(tokenizer.batch_decode(ft_outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Title: Efficacy of Antiviral Therapeutics in Severe Acute Respiratory Syndromes
Abstract: 2019 novel coronavirus (2019-nCoV) is a new type of coronavirus that causes severe acute respiratory syndrome (SARS). The first case of 2019-nCoV was reported in Wuhan, China, in December 2019. The virus has spread rapidly and has caused a global pandemic. The World Health Organization (WHO) has declared the outbreak of 2019-nCoV a public health emergency of international concern. The
